# 01 — Synthetic single-band field fit

This notebook fits a small **continuous spectral field** to a synthetic JWST-style patch on CPU in a few minutes.

**Prerequisite:** install an explicitly selected torchsky build into the harness `test` environment. Notebook execution uses the installed subject; it does not add a source checkout to `sys.path`. From the harness root, install the optional notebook UI once:

`pixi run -e test -- python -m pip install jupyterlab matplotlib`

Then launch from the harness root with `pixi run -e test -- jupyter lab notebooks/01_field_fit_synthetic.ipynb`. See the [torchsky field guide](https://github.com/astroai/torchsky/blob/main/docs/field.md).

In [ ]:
from __future__ import annotations

import torch
import matplotlib.pyplot as plt

from torchsky.map import (
    FieldObservation,
    estimate_field_domain,
    fit_field_model,
    make_field_model,
    render_field_observation,
)

torch.manual_seed(0)

header = {
    "NAXIS": 2,
    "NAXIS1": 32,
    "NAXIS2": 32,
    "CTYPE1": "RA---TAN",
    "CTYPE2": "DEC--TAN",
    "CRPIX1": 16.5,
    "CRPIX2": 16.5,
    "CRVAL1": 150.0,
    "CRVAL2": 2.0,
    "CDELT1": -0.05 / 3600.0,
    "CDELT2": 0.05 / 3600.0,
    "CUNIT1": "deg",
    "CUNIT2": "deg",
}

obs_meta = dict(
    spectral_min=5000.0,
    spectral_max=5000.0,
    transmission_center=5000.0,
)

template = FieldObservation(
    image=torch.zeros(32, 32, dtype=torch.float32),
    header=header,
    **obs_meta,
)

truth = make_field_model(
    "sine_implicit",
    estimate_field_domain([template]),
    hidden_dim=64,
    hidden_layers=3,
)
with torch.no_grad():
    target_image = render_field_observation(truth, template).detach()

obs = FieldObservation(image=target_image, header=header, **obs_meta)
domain = estimate_field_domain([obs])

torch.manual_seed(1)
model = make_field_model("sine_implicit", domain, hidden_dim=64, hidden_layers=3)

result = fit_field_model([obs], model, steps=120, lr=3e-3, loss_kind="charbonnier")
print("initial_loss", result.initial_loss, "final_loss", result.final_loss)

with torch.no_grad():
    pred = render_field_observation(model, obs)

fig, ax = plt.subplots(1, 3, figsize=(10, 3))
ax[0].imshow(target_image.numpy(), origin="lower")
ax[0].set_title("target")
ax[1].imshow(pred.numpy(), origin="lower")
ax[1].set_title("fit")
ax[2].imshow((pred - target_image).numpy(), origin="lower")
ax[2].set_title("residual")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()